# 베이스 모델 크기 실험 — Llama 3.1 8B / Qwen 2.5 7B
# Base Model Size Test — Llama 3.1 8B / Qwen 2.5 7B

**목적 | Purpose:**
지금까지 3B 모델이 정체성 학습에서 계속 붕괴(헛소리/다국어 섞임)한 원인이 모델 용량 한계일 수 있다는 가설을 확인.
Colab 무료 T4에서 7~8B급 모델이 4비트로 로드되는지, 기본 동작이 정상인지만 확인한다 (파인튜닝은 안 함).

**순서 | Steps:**
1. 설치
2. 8B 모델 로드 시도 (실패하면 7B로 대체)
3. GPU 메모리 사용량 확인
4. 간단한 질문 1~2개로 기본 동작 확인 (원본 모델, 파인튜닝 전)


In [ ]:
# 셀 1: 필수 라이브러리 설치
# Cell 1: Install required libraries
!pip install unsloth


In [ ]:
# 셀 2: CUDA 오류 방지 + 8B 모델 로드 시도
# Cell 2: CUDA fix + try loading 8B model
import ctypes, glob

paths = (
    glob.glob("/usr/local/lib/python*/dist-packages/nvidia/*/lib/libnvJitLink*.so*") +
    glob.glob("/usr/lib/**/libnvJitLink*.so*", recursive=True) +
    glob.glob("/usr/local/cuda*/**/libnvJitLink*.so*", recursive=True)
)
if paths:
    ctypes.CDLL(paths[0], mode=ctypes.RTLD_GLOBAL)
    print(f"CUDA fix 적용 | CUDA fix applied: {paths[0]}")

from unsloth import FastLanguageModel
import torch

MODEL_NAME = "unsloth/Meta-Llama-3.1-8B-Instruct"  # 실패하면 "unsloth/Qwen2.5-7B-Instruct"로 교체

try:
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=MODEL_NAME,
        max_seq_length=512,
        load_in_4bit=True,
    )
    print(f"모델 로드 성공 | Model loaded successfully: {MODEL_NAME}")
except Exception as e:
    print(f"로드 실패 | Load failed: {e}")
    print("Qwen 2.5 7B로 재시도 | Retrying with Qwen 2.5 7B...")
    MODEL_NAME = "unsloth/Qwen2.5-7B-Instruct"
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=MODEL_NAME,
        max_seq_length=512,
        load_in_4bit=True,
    )
    print(f"모델 로드 성공 | Model loaded successfully: {MODEL_NAME}")


In [ ]:
# 셀 3: GPU 메모리 사용량 확인
# Cell 3: Check GPU memory usage
import torch

allocated = torch.cuda.memory_allocated() / 1e9
reserved = torch.cuda.memory_reserved() / 1e9
total = torch.cuda.get_device_properties(0).total_memory / 1e9

print(f"사용 중 | Allocated: {allocated:.2f} GB")
print(f"예약됨 | Reserved: {reserved:.2f} GB")
print(f"전체 GPU 메모리 | Total GPU memory: {total:.2f} GB")
print(f"여유 | Free: {total - reserved:.2f} GB")


In [ ]:
# 셀 4: 기본 동작 확인 (파인튜닝 전 원본 모델)
# Cell 4: Basic sanity check (original model, no fine-tuning)
FastLanguageModel.for_inference(model)

test_questions = [
    "안녕? 자기소개 해줘.",
    "LoRA가 뭐야?",
]
for q in test_questions:
    inputs = tokenizer([q], return_tensors="pt").to("cuda")
    outputs = model.generate(**inputs, max_new_tokens=100, temperature=0.2)
    print(tokenizer.decode(outputs[0], skip_special_tokens=True))
    print("---")

print(f"\n오늘 결론에 기록할 것: 사용한 모델={MODEL_NAME}, 로드 성공 여부, 메모리 사용량, 답변이 정상적인 문장인지")
